In [ ]:
import os, json, requests
import numpy as np

stats_dir = 'stats'
pdb_dir = '/mnt/hdd/yenlin/data/Protein_Data_Bank'
pdb_download_dir = f'{pdb_dir}/pdb.gz'


# PARAMETERS
coverage_mode = 0
# similarity_cutoff = 1
similarity_cutoff = 0.95

tag = f'sim_{similarity_cutoff}-annotated'


In [ ]:
### READ DATA FROM PREVIOUS SCRIPT

# META DATA
all_info = np.loadtxt(
    f'{stats_dir}/OUT-2-2.all_information-sim_{similarity_cutoff}.csv',
    delimiter=',',
    dtype=np.str_,
    comments=None
)

header = all_info[0].tolist()
header[0] = header[0][2:]
all_info = all_info[1:]

print(header)

# FASTA
with open(
    f'{stats_dir}/OUT-2-2.all_sequences-sim_{similarity_cutoff}.fasta',
    'r'
) as f:
    all_sequences = f.read().split('>')[1:]
all_sequences = np.array([
    entry.split('\n', 1)[1].replace('\n', '') for entry in all_sequences
])

print(all_sequences[1])

assert len(all_info) == len(all_sequences)
print()
print(len(all_info), 'entries')


['full_id', 'pdb_id', 'assembly_id', 'multimericity', 'auth_chain_id', 'asym_chain_id', 'seq_database', 'seq_database_accession']
AAAAANHSTQESGFDYEGLIDSELQKKRLDKSYRYFNNINRLAKEFPLAHRQREADKVTVWCSNDYLALSKHPEVLDAMHKTIDKYGCGAGGTRNIAGHNIPTLNLEAELATLHKKEGALVFSSCYVANDAVLSLLGQKMKDLVIFSDELNHASMIVGIKHANVKKHIFKHNDLNELEQLLQSYPKSVPKLIAFESVYSMAGSVADIEKICDLADKYGALTFLDEVHAVGLYGPHGAGVAEHCDFESHRASGIATPKTNDKGGAKTVMDRVDMITGTLGKSFGSVGGYVAASRKLIDWFRSFAPGFIFTTTLPPSVMAGATAAIRYQRCHIDLRTSQQKHTMYVKKAFHELGIPVIPNPSHIVPVLIGNADLAKQASDILINKHQIYVQAINFPTVARGTERLRITPTPGHTNDLSDILINAVDDVFNELQLPRVRDWESQGGLLGVGESGFVEESNLWTSSQLSLTNDDLNPNVRDPIVKQLEVSSGIKQ

12534 entries


In [6]:
### DOWNLOAD ANNOTATIONS FROM RCSB PDB

# read template data API payload
data_api_payload_file = 'template-data_api-go_annotations.txt'

with open(data_api_payload_file, 'r') as f:
    data_api_template = f.read()

payload = data_api_template.replace(
    'ASSEMBLY_LIST_PLACEHOLDER',
    '["' + '","'.join(all_info[:,header.index('full_id')]) + '"]'
)

url = 'https://data.rcsb.org/graphql'

response = requests.post(url, json={'query': payload})

if response.status_code != 200:
    print(response.text)
decoded = response.json()


In [ ]:
### FILTER OUT GO ANNOTATIONS

decoded_filtered = {}

entries_to_keep = []
entries_with_multiple_polymer_entities = []
entries_without_annotations = []

for entry_idx, entry in enumerate(decoded['data']['assemblies']):
    entry_id = entry['rcsb_id']

    if len(entry['entry']['polymer_entities']) != 1:
        entries_with_multiple_polymer_entities.append(entry_id)
        continue
    if entry['entry']['polymer_entities'][0]['rcsb_polymer_entity_annotation'] is None:
        entries_without_annotations.append(entry_id)
        continue

    terms = [
        a for a in entry['entry']['polymer_entities'][0]['rcsb_polymer_entity_annotation']
        if a['type'] == 'GO'
    ]
    if len(terms) == 0:
        entries_without_annotations.append(entry_id)
        continue

    decoded_filtered[entry_id] = terms
    entries_to_keep.append(entry_id)

print(f'# entries with more than one polymer entity: {len(entries_with_multiple_polymer_entities)}')
print(f'# entries with no GO annotations: {len(entries_without_annotations)}')

assert len(decoded_filtered) == len(entries_to_keep)
print(f'# entries with GO annotations: {len(entries_to_keep)}')

# SAVE SNAPSHOT
np.savetxt(
    f'{stats_dir}/OUT-3-1.ids_without_annotations.txt',
    entries_without_annotations,
    fmt='%s'
)
with open(f'{stats_dir}/OUT-3-1.all_annotations.json', 'w') as f:
    json.dump(decoded, f, indent=2)

# SUMMARY
loc = np.isin(all_info[:,header.index('full_id')], entries_to_keep)
all_info_remaining = all_info[loc]
all_sequences_remaining = all_sequences[loc]
assert len(all_info_remaining) == len(entries_to_keep)

_, cnt = np.unique(
    all_info_remaining[:, header.index('multimericity')],
    return_counts=True
)
print(cnt, np.sum(cnt))


# entries with more than one polymer entity: 0
# entries with no GO annotations: 1175
# entries with GO annotations: 11359
[8231  884 1666   81  497] 11359


In [8]:
### SAVE INTO

np.savetxt(
    f'{stats_dir}/OUT-3-2.all_information-{tag}.csv',
    all_info_remaining,
    delimiter=',',
    fmt='%s',
    header=','.join(header)
)

# FASTA
all_auth_chain_id_remaining = all_info_remaining[:,header.index('auth_chain_id')]
all_asym_chain_id_remaining = all_info_remaining[:,header.index('asym_chain_id')]
with open(f'{stats_dir}/OUT-3-2.all_sequences-{tag}.fasta', 'w') as f:
    for assembly_idx in range(len(all_sequences_remaining)):
        f.write(
            f'>{all_info_remaining[assembly_idx,header.index("full_id")]}'
            f'auth_chain_id:{all_auth_chain_id_remaining[assembly_idx]},'
            f'asym_chain_id:{all_asym_chain_id_remaining[assembly_idx]}\n'
        )
        f.write(f'{all_sequences_remaining[assembly_idx]}\n')


In [10]:
### BUILD CONCISE LIST OF GO ANNOTATIONS (SORTED BY GO NUMBER)

go_num2name = {}
go_lineage = {}

for entry_id, annotations in decoded_filtered.items():
    go_lineage[entry_id] = []

    for annotation in annotations:
        lineage = annotation['annotation_lineage']
        for term in lineage:
            go_lineage[entry_id].append(term['id'])
            go_num2name[term['id']] = term['name']

    go_lineage[entry_id] = np.unique(go_lineage[entry_id]).tolist()

go_num2name_sorted = dict(sorted(go_num2name.items()))

print(len(go_lineage))

# GET OCCURRENCES
go_count = {}

for entry_id, terms in go_lineage.items():
    for term in terms:
        go_count[term] = go_count.get(term, 0) + 1

go_count_sorted = dict(sorted(go_count.items()))

# SAVE
with open(f'{stats_dir}/OUT-3-2.go_annotations.json', 'w') as f:
    json.dump(go_lineage, f, indent=2)

# merge num2names and count as array
n2n_count = np.array(
    [
        [k, go_count_sorted[k], go_num2name_sorted[k]]
        for k in go_num2name_sorted.keys()
    ],
    dtype=np.str_
)

np.savetxt(
    f'{stats_dir}/OUT-3-2.go_num2names_and_count.txt',
    n2n_count,
    fmt='%s %5s "%s"'
)
np.savetxt(
    f'{stats_dir}/OUT-3-2.go_num2names_and_count-sorted.txt',
    n2n_count[np.argsort(n2n_count[:, 1].astype(np.int_))[::-1]],
    fmt='%s %5s "%s"'
)

11359
